In [ ]:
from google.colab import drive
drive.mount("/content/drive")

REPO_URL = "https://github.com/m1ksj/sonar-mine-detection.git"
REPO_DIR = "/content/sonar-mine-detection"

DRIVE_DIR = "/content/drive/Shareddrives/AML_SSS_GROUP"
DATA_ZIP = f"{DRIVE_DIR}/darknet_data.zip"

RUN_NAME = "yolo26n_cv_sss_aug_45runs_v1"

CV_OUT = f"{DRIVE_DIR}/experiments/{RUN_NAME}"
CV_WORK = "/content/yolo26n_cv_work"

DEBUG_OUT = f"{DRIVE_DIR}/experiments/{RUN_NAME}_debug"
DEBUG_WORK = "/content/yolo26n_debug_work"

GITHUB_SAFE_OUT = f"{DRIVE_DIR}/github_safe_artifacts/{RUN_NAME}"

DEVICE = 0

In [ ]:
%cd /content

!rm -rf "{REPO_DIR}"
!git clone --branch dev "{REPO_URL}" "{REPO_DIR}"

%cd "{REPO_DIR}"

!pip install -q ultralytics==8.4.50 pandas pyyaml matplotlib

!git log --oneline --decorate -5
!python -c "import ultralytics; print('ultralytics', ultralytics.__version__)"
!nvidia-smi

In [ ]:
%cd "{REPO_DIR}"

from pathlib import Path

!mkdir -p data/processed
!rm -rf data/processed/darknet
!unzip -q "{DATA_ZIP}" -d data/processed

DATA_ROOT = f"{REPO_DIR}/data/processed/darknet"

jpg_count = len(list(Path(DATA_ROOT).rglob("*.jpg")))
txt_count = len(list(Path(DATA_ROOT).rglob("*.txt")))

print("DATA_ROOT:", DATA_ROOT)
print("jpg:", jpg_count)
print("txt:", txt_count)

print("\nTrain images:")
!find data/processed/darknet/train -name "*.jpg" | wc -l

print("\nVal images:")
!find data/processed/darknet/val -name "*.jpg" | wc -l

print("\nTest images:")
!find data/processed/darknet/test -name "*.jpg" | wc -l

assert jpg_count == 1170, "Expected 1170 jpg files after unzip."
assert txt_count >= 1170, "Expected label txt files after unzip."

In [ ]:
%cd "{REPO_DIR}"

import pandas as pd
from pathlib import Path
import yaml

split_dir = Path("data/splits")

print("Main split counts:")
for name in ["train", "val", "test"]:
    print(name, len(pd.read_csv(split_dir / f"{name}.csv")))

print("\nCV fold counts:")
for fold in range(5):
    train_n = len(pd.read_csv(split_dir / f"fold_{fold}_train.csv"))
    val_n = len(pd.read_csv(split_dir / f"fold_{fold}_val.csv"))
    print(f"fold_{fold}: train={train_n}, val={val_n}")

with open("configs/yolo26n_tuning.yaml", "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

n_runs = (
    len(cfg["search_space"]["augmentation_mode"])
    * len(cfg["search_space"]["lr0"])
    * len(cfg["search_space"]["batch"])
    * len(cfg["search_space"]["patience"])
    * len(cfg["search_space"]["optimizer"])
    * len(cfg["cv"]["folds"])
)

print("\nPlanned CV runs:", n_runs)
print("Augmentation modes:", cfg["search_space"]["augmentation_mode"])
print("Learning rates:", cfg["search_space"]["lr0"])
print("Batch:", cfg["search_space"]["batch"])
print("Patience:", cfg["search_space"]["patience"])
print("Optimizer:", cfg["search_space"]["optimizer"])

assert n_runs == 45, "Expected 45 planned CV runs."

In [ ]:
%cd "{REPO_DIR}"

!python scripts/train_yolo26n_cv.py \
  --config configs/yolo26n_tuning.yaml \
  --output-dir /content/yolo26n_cv_dryrun \
  --data-root "{DATA_ROOT}" \
  --dry-run

In [ ]:
import subprocess
from pathlib import Path

def run_and_log(cmd, log_path):
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)

    print("Running command:")
    print(" ".join(cmd))
    print("\nLog file:", log_path)

    with open(log_path, "w", encoding="utf-8") as log_file:
        process = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )

        for line in process.stdout:
            print(line, end="")
            log_file.write(line)

        return_code = process.wait()

    assert return_code == 0, f"Command failed with exit code {return_code}"

In [ ]:
%cd "{REPO_DIR}"

Path(CV_OUT).mkdir(parents=True, exist_ok=True)

full_cmd = [
    "python", "scripts/train_yolo26n_cv.py",
    "--config", "configs/yolo26n_tuning.yaml",
    "--output-dir", CV_OUT,
    "--work-dir", CV_WORK,
    "--data-root", DATA_ROOT,
    "--device", str(DEVICE),
    "--workers", "8",
    "--train-final",
]

run_and_log(full_cmd, f"{CV_OUT}/full_cv_console.log")

In [ ]:
import pandas as pd
from pathlib import Path

cv_out = Path(CV_OUT)

results_path = cv_out / "tables" / "yolo26n_cv_results.csv"
summary_path = cv_out / "tables" / "yolo26n_cv_summary.csv"

results = pd.read_csv(results_path)
summary = pd.read_csv(summary_path)

print("CV results path:", results_path)
print("CV summary path:", summary_path)

print("\nCV results shape:", results.shape)
print("CV summary shape:", summary.shape)

display(
    summary.sort_values("map50_95_mean", ascending=False).head(15)
)

In [ ]:
top_configs = summary.sort_values("map50_95_mean", ascending=False).head(15)

top_config_path = cv_out / "tables" / "yolo26n_top_cv_configs.csv"
top_configs.to_csv(top_config_path, index=False)

print("Saved:", top_config_path)
display(top_configs)

In [ ]:
from pathlib import Path

selected_path = Path(CV_OUT) / "final_selected_config.yaml"

print("Selected config path:")
print(selected_path)

print("\nSelected config:")
print(selected_path.read_text())

In [ ]:
from pathlib import Path

runs_dir = Path(CV_OUT) / "ultralytics_runs"

print("Run folders:")
for p in sorted(runs_dir.glob("*")):
    print(p.name)

final_run = runs_dir / "final_yolo26n_best_cv_config"
test_eval = runs_dir / "final_yolo26n_test_eval"

print("\nFinal run exists:", final_run.exists())
print("Test eval exists:", test_eval.exists())

print("\nFinal run files:")
!find "{final_run}" -maxdepth 2 -type f | sort | head -80

print("\nTest eval files:")
!find "{test_eval}" -maxdepth 2 -type f | sort | head -80

In [ ]:
import yaml
from pathlib import Path

final_args_path = Path(CV_OUT) / "ultralytics_runs" / "final_yolo26n_best_cv_config" / "args.yaml"

print(final_args_path)

with open(final_args_path, "r", encoding="utf-8") as f:
    final_args = yaml.safe_load(f)

keys = [
    "data", "model", "imgsz", "epochs", "batch", "optimizer", "lr0", "patience",
    "mosaic", "close_mosaic", "mixup", "cutmix", "copy_paste",
    "degrees", "translate", "scale", "shear", "perspective",
    "fliplr", "flipud", "hsv_h", "hsv_s", "hsv_v", "bgr", "erasing",
]

for key in keys:
    print(f"{key}: {final_args.get(key)}")

In [ ]:
from pathlib import Path
import pandas as pd
from ultralytics import YOLO

cv_out = Path(CV_OUT)

final_weights = (
    cv_out
    / "ultralytics_runs"
    / "final_yolo26n_best_cv_config"
    / "weights"
    / "best.pt"
)

final_dataset_yaml = (
    Path(CV_WORK)
    / "final_train_val_test"
    / "data.yaml"
)

model = YOLO(str(final_weights))

metrics = model.val(
    data=str(final_dataset_yaml),
    split="test",
    imgsz=640,
    batch=16,
    device=DEVICE,
    workers=4,
    project=str(cv_out / "ultralytics_runs"),
    name="final_yolo26n_test_eval_saved_metrics",
    plots=True,
)

test_metrics = {
    "precision": float(metrics.box.mp),
    "recall": float(metrics.box.mr),
    "map50": float(metrics.box.map50),
    "map50_95": float(metrics.box.map),
}

test_metrics_path = cv_out / "tables" / "final_yolo26n_test_metrics.csv"
pd.DataFrame([test_metrics]).to_csv(test_metrics_path, index=False)

print("Saved:", test_metrics_path)
print(pd.DataFrame([test_metrics]).to_string(index=False))

In [ ]:
import shutil
from pathlib import Path

cv_out = Path(CV_OUT)
safe_out = Path(GITHUB_SAFE_OUT)

if safe_out.exists():
    shutil.rmtree(safe_out)

(safe_out / "results_tables").mkdir(parents=True, exist_ok=True)
(safe_out / "configs").mkdir(parents=True, exist_ok=True)
(safe_out / "figures" / "final_training").mkdir(parents=True, exist_ok=True)
(safe_out / "figures" / "test_evaluation").mkdir(parents=True, exist_ok=True)
(safe_out / "logs").mkdir(parents=True, exist_ok=True)

for path in (cv_out / "tables").glob("*.csv"):
    shutil.copy2(path, safe_out / "results_tables" / path.name)

for name in [
    "used_yolo26n_tuning.yaml",
    "used_augmentation_yolo26n.yaml",
    "final_selected_config.yaml",
]:
    src = cv_out / name
    if src.exists():
        shutil.copy2(src, safe_out / "configs" / name)

for name in ["full_cv_console.log"]:
    src = cv_out / name
    if src.exists():
        shutil.copy2(src, safe_out / "logs" / name)

final_run = cv_out / "ultralytics_runs" / "final_yolo26n_best_cv_config"
test_eval = cv_out / "ultralytics_runs" / "final_yolo26n_test_eval_saved_metrics"

for name in ["args.yaml", "results.csv", "results.png"]:
    src = final_run / name
    if src.exists():
        shutil.copy2(src, safe_out / "results_tables" / f"final_train_{name}")

for pattern in ["*.png", "*.jpg"]:
    for src in final_run.glob(pattern):
        shutil.copy2(src, safe_out / "figures" / "final_training" / src.name)

    for src in test_eval.glob(pattern):
        shutil.copy2(src, safe_out / "figures" / "test_evaluation" / src.name)

for name in ["args.yaml", "results.csv", "results.png"]:
    src = test_eval / name
    if src.exists():
        shutil.copy2(src, safe_out / "results_tables" / f"test_eval_{name}")

print("GitHub-safe artifacts saved to:")
print(safe_out)

print("\nFiles:")
!find "{safe_out}" -type f | sort

In [ ]:
from pathlib import Path

cv_out = Path(CV_OUT)

final_best = cv_out / "ultralytics_runs" / "final_yolo26n_best_cv_config" / "weights" / "best.pt"
final_last = cv_out / "ultralytics_runs" / "final_yolo26n_best_cv_config" / "weights" / "last.pt"

note_path = Path(GITHUB_SAFE_OUT) / "final_model_locations.txt"

note = f"""Final YOLO26n model weights are stored in Drive, not in Git.

Best weights:
{final_best}

Last weights:
{final_last}

Full experiment folder:
{CV_OUT}

GitHub-safe artifacts:
{GITHUB_SAFE_OUT}
"""

note_path.write_text(note, encoding="utf-8")

print(note)

In [ ]:
print("Full experiment folder:")
!du -sh "{CV_OUT}"

print("\nGitHub-safe artifact folder:")
!du -sh "{GITHUB_SAFE_OUT}"

print("\nMain tables:")
!find "{CV_OUT}/tables" -maxdepth 1 -type f | sort

print("\nFinal selected config:")
!cat "{CV_OUT}/final_selected_config.yaml"

print("\nFinal model weights:")
!find "{CV_OUT}/ultralytics_runs/final_yolo26n_best_cv_config/weights" -type f | sort